# Colab 02 — One baseline experiment

## Goal

Stage one dataset, run one explicitly configured GCN/GAT/GIN baseline experiment, and copy its checkpoint plus lightweight records back to Drive.

**Training and synchronisation default to off.**


## Setup


In [17]:
!git clone --depth 1 \
    --branch v2-refactor \
    --single-branch \
    https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git \
    /content/msc-dissertation-drug-repurposing

%cd /content/drive/MyDrive/msc-dissertation-drug-repurposing

!pip install -q -r requirements.txt

from google.colab import drive
drive.mount("/content/drive")

Cloning into '/content/msc-dissertation-drug-repurposing'...
fatal: could not read Username for 'https://github.com': No such device or address
/content/drive/MyDrive/msc-dissertation-drug-repurposing
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from pathlib import Path
from importlib import import_module
import json
import os
import shutil
import subprocess
import sys

GIT_URL = "https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git"
BRANCH = "v2-refactor"
REPO_DIR = Path("/content/msc-dissertation-drug-repurposing")
WORKSPACE_DIR = Path("/content/dti_workspace")
DRIVE_ROOT = Path("/content/drive/MyDrive/msc-dissertation-drug-repurposing")


def configure_workspace_environment() -> None:
    # Configure all paths before importing config or any src module.
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(f"Missing required v2 API: {module_name}.{attribute}")
    return getattr(module, attribute)


def copy_directory(source: Path, destination: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)


print(json.dumps({
    "git_url": GIT_URL,
    "branch": BRANCH,
    "repo": str(REPO_DIR),
    "workspace": str(WORKSPACE_DIR),
    "drive_root": str(DRIVE_ROOT),
}, indent=2))


{
  "git_url": "https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git",
  "branch": "v2-refactor",
  "repo": "/content/msc-dissertation-drug-repurposing",
  "workspace": "/content/dti_workspace",
  "drive_root": "/content/drive/MyDrive/msc-dissertation-drug-repurposing"
}


In [19]:
DATASET = "davis"       # davis | kiba
SPLIT = "random"        # random | cold_drug | cold_target
MODEL = "gcn"           # gcn | gat | gin
SEED = 42               # 42 | 123 | 2026

RUN_STAGE_DATA = False
RUN_TRAINING = False
RUN_SYNC_TO_DRIVE = False


In [20]:
DATASET = "davis"
SPLIT = "random"
MODEL = "gin"
SEED = 42

RUN_STAGE_DATA = True
RUN_TRAINING = True
RUN_SYNC_TO_DRIVE = True

## Steps


In [21]:
if RUN_STAGE_DATA:
    copy_directory(
        DRIVE_ROOT / "data" / "raw" / DATASET,
        WORKSPACE_DIR / "data" / "raw" / DATASET,
    )
    if (REPO_DIR / "data" / "splits").exists():
        copy_directory(REPO_DIR / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
    if (DRIVE_ROOT / "data" / "splits").exists():
        copy_directory(DRIVE_ROOT / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
else:
    print("Data staging skipped: RUN_STAGE_DATA is False.")


In [22]:
import json
import pandas as pd

from config import processed_pairs_path
from src.preprocessing import (
    ensure_pair_ids,
    scientific_pairs_sha256,
)

DATASET = "davis"

csv_path = processed_pairs_path(DATASET)
metadata_path = csv_path.with_suffix(".metadata.json")

print("CSV:", csv_path)
print("Metadata:", metadata_path)

if not csv_path.exists():
    raise FileNotFoundError(csv_path)

if not metadata_path.exists():
    raise FileNotFoundError(metadata_path)

pairs = pd.read_csv(
    csv_path,
    dtype={
        "dataset": str,
        "pair_id": str,
        "drug_id": str,
        "target_id": str,
        "smiles": str,
        "protein_sequence": str,
        "affinity_type": str,
    },
)

pairs = ensure_pair_ids(pairs)

with metadata_path.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

old_hash = metadata["scientific_data_sha256"]
new_hash = scientific_pairs_sha256(pairs)

print("Old fingerprint:", old_hash)
print("New fingerprint:", new_hash)
print("Match before repair:", old_hash == new_hash)

metadata["scientific_data_sha256"] = new_hash

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
    file.write("\n")

print("\nFingerprint metadata repaired.")

CSV: /content/drive/MyDrive/msc-dissertation-drug-repurposing/data/processed/davis_pairs.csv
Metadata: /content/drive/MyDrive/msc-dissertation-drug-repurposing/data/processed/davis_pairs.metadata.json
Old fingerprint: 4e6ed96dddf38df38da139448891611f0676e66424a86529319a79768ca1f96f
New fingerprint: 4e6ed96dddf38df38da139448891611f0676e66424a86529319a79768ca1f96f
Match before repair: True

Fingerprint metadata repaired.


In [23]:
experiment_record = None
if RUN_TRAINING:
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    from config import ExperimentConfig, SPLITS_DIR
    run_baseline_experiment = require_api("src.training", "run_baseline_experiment")
    experiment = ExperimentConfig(
        dataset=DATASET, split=SPLIT, model=MODEL, seed=SEED
    )
    manifest_path = SPLITS_DIR / "manifests" / (
        f"{experiment.dataset}_{experiment.split}_seed{experiment.seed}_manifest.csv"
    )
    experiment_record = run_baseline_experiment(
        config=experiment,
        split_manifest_path=manifest_path,
    )
    print(json.dumps(experiment_record, indent=2, default=str))
else:
    print("Baseline training skipped: RUN_TRAINING is False.")


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 001 | Train MSE: 1.1341 | Validation MSE: 0.6781 | RMSE: 0.8235 | MAE: 0.4981
Epoch 002 | Train MSE: 0.9600 | Validation MSE: 1.1760 | RMSE: 1.0844 | MAE: 0.7665
Epoch 003 | Train MSE: 0.9232 | Validation MSE: 0.6375 | RMSE: 0.7984 | MAE: 0.5538
Epoch 004 | Train MSE: 0.9007 | Validation MSE: 0.9999 | RMSE: 0.9999 | MAE: 0.6682
Epoch 005 | Train MSE: 0.8925 | Validation MSE: 0.7219 | RMSE: 0.8496 | MAE: 0.5175
Epoch 006 | Train MSE: 0.8838 | Validation MSE: 0.7365 | RMSE: 0.8582 | MAE: 0.5298
Epoch 007 | Train MSE: 0.8614 | Validation MSE: 0.6416 | RMSE: 0.8010 | MAE: 0.5951
Epoch 008 | Train MSE: 0.8406 | Validation MSE: 0.8683 | RMSE: 0.9318 | MAE: 0.6180
Epoch 009 | Train MSE: 0.8549 | Validation MSE: 0.6139 | RMSE: 0.7835 | MAE: 0.5457
Epoch 010 | Train MSE: 0.8246 | Validation MSE: 0.8365 | RMSE: 0.9146 | MAE: 0.5693
Epoch 011 | Train MSE: 0.8191 | Validation MSE: 0.7931 | RMSE: 0.8906 | MAE: 0.5558
Epoch 012 | Train MSE: 0.8145 | Validation MSE: 0.6958 | RMSE: 0.8342 | MAE:

In [26]:
if RUN_SYNC_TO_DRIVE:
    if experiment_record is None:
        raise RuntimeError("Run and inspect the experiment before syncing.")
    for name in ("checkpoints", "results"):
        source = WORKSPACE_DIR / name
        if source.exists():
            copy_directory(source, DRIVE_ROOT / name)
    if (WORKSPACE_DIR / "data" / "splits").exists():
        copy_directory(
            WORKSPACE_DIR / "data" / "splits",
            DRIVE_ROOT / "data" / "splits",
        )
    print("Durable outputs synced to:", DRIVE_ROOT)
else:
    print("Drive sync skipped: RUN_SYNC_TO_DRIVE is False.")


Durable outputs synced to: /content/drive/MyDrive/msc-dissertation-drug-repurposing


## Checks

Verify validation-MSE checkpoint selection, dataset-specific score labels, final test isolation, and the full structured experiment record before synchronising.

## Next steps

Change only one dataset/split/model/seed configuration at a time.
